
# Model Training & Evaluation Pipeline with LoRA + 4-bit Quantization

This repository contains a full pipeline for **multi-class text classification** using a transformer model (e.g., LLaMA 3.1 8B by default) with **LoRA (parameter-efficient fine-tuning)** and **4-bit quantization** (BitsAndBytes). It implements a **cyclic 5-fold scheme** (validation = fold *r*, test = fold *(r+1) % 5*), early stopping on validation **loss**, best-model selection by **Macro\_F1**, and optional **temperature scaling** saved per fold and for the final model.

## Overview

1. **Data Loading & Preprocessing**

   * Loads a CSV with columns `comment` and `labels`.
   * Maps string labels to **integer class IDs** and logs a **class distribution** summary.

2. **Model & Tokenizer**

   * Loads a pretrained model with **4-bit quantization** via `BitsAndBytesConfig` (`nf4`, double quant, `bfloat16` compute).
   * Configures padding token (uses EOS as PAD to avoid padding errors).

3. **LoRA Integration**

   * Applies LoRA adapters (targets `q_proj`, `k_proj`, `v_proj`; in test mode only `q_proj`, `v_proj`) and prepares the model for k-bit training.

4. **Cyclic Cross-Validation (5 folds)**

   * For each round **r**:

     * **val = fold r**, **test = fold (r+1) % 5**, **train** = the remaining 3 folds (union of their original `test_idx`).
     * Trains with **early stopping on validation loss** (patience = 3).
     * Selects best checkpoint by **Macro\_F1** (even though early stop is driven by loss).
     * Evaluates on the **test** split and appends metrics to `Performance_Detailed.csv`.
   * After all rounds, appends a **“Mean±95%”** row (mean ± 95% CI across folds).

5. **Final Model Training**

   * Performs a fresh **stratified split** (default 80% train / 20% val).
   * Retrains the final model, evaluates and appends a **Final-Model** row to the CSV, and saves the model and tokenizer to `./saved_model`.

6. **Temperature Scaling (Optional, Included)**

   * Fits a scalar temperature **T** on the validation set by minimizing NLL (LBFGS over `logT`).
   * Saves `temperature.json` per fold (e.g., `./results/fold_r0/temperature.json`) and for the **final model** (`./saved_model/temperature.json`).

## Key Features

* **LoRA** for efficient fine-tuning on large models.
* **4-bit quantization** with `nf4` + double quantization, compute in `bfloat16`.
* **Class-weighted loss** (inverse frequency, normalized) with **label smoothing** (0.05).
* **Cyclic 5-fold CV** where each fold is used once as **validation** and once as **test**.
* **Early stopping** on validation **loss** (patience=3, `min_delta=1e-4`).
* **Best model selection** by **Macro\_F1** at epoch level.
* **Cosine LR schedule**, `warmup_ratio=0.05`, gradient accumulation = 2.
* Mixed precision: **fp16** (training) and **tf32** enabled.
* Results appended to **`Performance_Detailed.csv`** with:

  * One row per round (`fold=r0..r4`)
  * A **“Mean±95%”** summary row
  * A **“Final-Model”** row
* **Batch inference** on evaluation to reduce memory spikes.

## Metrics Logged

For each evaluation, the pipeline records:

* **Accuracy**, **Balanced Accuracy**
* **Per-class** Precision / Recall / F1
* **Macro** Precision / Recall / F1
  (Per-fold rows use the scikit-learn `classification_report` dictionary; the CSV flattens those keys.)

## Training Details (as implemented)

* **Tokenizer max length**: `MAX_LENGTH = 128` (96 in test mode).
* **Batch size**: `BATCH_SIZE = 128` (1 in test mode).
* **Epochs**: up to `N_EPOCHS = 50` (3 in test mode) with early stopping.
* **Optimizer**: `adamw_torch`, `weight_decay=0.01`, `max_grad_norm=1.0`.
* **DataLoader workers**: `dataloader_num_workers=16`.
* **Gradient checkpointing**: enabled to save memory.
* **Random seeds**: multiple sources seeded (`numpy`, `random`, `torch`, CUDA). CuDNN set to deterministic.

## Reproducibility

* Seeds are fixed (`SEED=1991`) and CuDNN is deterministic.
* `CUDA_LAUNCH_BLOCKING=1` aids debugging; memory config via `PYTORCH_CUDA_ALLOC_CONF=max_split_size_mb:32`.
* `WANDB_DISABLED=true` and `TOKENIZERS_PARALLELISM=false` to reduce noise/warnings.

## Inputs & Outputs

* **Input CSV**: `FinalLabels.csv` with columns:

  * `comment` (text)
  * `labels` (string label; mapped to int)
* **Outputs**:

  * `./results/fold_r*/` — checkpoints + `temperature.json` per round.
  * `Performance_Detailed.csv` — per-round metrics, **Mean±95%**, and **Final-Model**.
  * `./saved_model/` — final model, tokenizer, and `temperature.json`.

## Requirements

* `transformers`, `datasets`, `peft`, `scikit-learn`, `torch`, `pandas`, `tqdm`, `scipy`, `bitsandbytes` (via transformers’ quantization).

> Tip: Ensure CUDA-compatible builds of PyTorch and BitsAndBytes for 4-bit.

## Usage (high level)

1. Place `FinalLabels.csv` in the working directory with `comment` and `labels`.
2. (Optional) Set `FLAG_TEST=True` for a fast dry run (smaller model, fewer folds/epochs).
3. Run the script. It will:

   * Check CUDA, load data, map labels.
   * Run **cyclic 5-fold CV** with LoRA + 4-bit.
   * Append metrics to `Performance_Detailed.csv` (including **Mean±95%**).
   * Retrain and save the **final model** to `./saved_model`.

## Notes & Caveats

* **Token management**: the code uses a hardcoded `hf_token`. For security, consider moving it to an environment variable (e.g., `HF_TOKEN`) and passing it to `from_pretrained(..., token=os.getenv("HF_TOKEN"))`.
* **Hardware**: 4-bit quantization reduces memory, but you still need a GPU with sufficient VRAM for the chosen model (`meta-llama/Meta-Llama-3.1-8B` by default). Use `FLAG_TEST=True` to verify the pipeline quickly with `TinyLlama`.
* **Validation vs. Selection**: Early stopping is based on **loss** (smoother signal), while the best checkpoint for saving is chosen by **Macro\_F1**—this is intentional to balance stability and task objective.



In [ ]:
# -*- coding: utf-8 -*-
"""
Multilabel trainer (2 binary labels):
- Input columns: 'comment', 'Autorrelato', 'Sintomas / Efeitos Colaterais'
- Multilabel vector per sample: [Autorrelato, Sintomas]
- Trainer uses BCEWithLogits + pos_weight
- Multilabel stratified CV (MSKF) with cyclic scheme (val=r, test=r+1)
- LoRA on top of a 4-bit quantized base model
- Micro/macro metrics + per-label metrics with human-friendly names
- Per-label threshold tuning via validation F1 (saved and reused)
"""

# --- Built-ins
import os
import gc
import logging
import warnings
import json
import random
import shutil

# --- Third-party: numerical & utility libraries
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy import stats

# --- PyTorch 
import torch
from torch import nn
from torch.utils.data import DataLoader
import torch.optim as optim
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

import torchvision
torchvision.disable_beta_transforms_warning()

from datasets import Dataset
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

from sklearn.metrics import precision_recall_fscore_support, precision_recall_curve

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig,
    TrainingArguments, Trainer, default_data_collator, TrainerCallback,
    EarlyStoppingCallback, EvalPrediction, set_seed,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# --------------------------------------------------------------------
# Global configs / general settings
# --------------------------------------------------------------------
torchvision.disable_beta_transforms_warning()
os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Avoid tokenizer warning
os.environ["WANDB_DISABLED"] = "true"          # Disable W&B
os.environ['CUDA_LAUNCH_BLOCKING'] = "0"       # Better CUDA debug
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:32"

# suppress specific future warnings
warnings.filterwarnings(
    "ignore",
    message="`torch.cpu.amp.autocast.*` is deprecated",
    category=FutureWarning,
)

# Logger
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger("multilabel-trainer")

# Token Hugging Face (use environment variable; do not hardcode)
hf_token = os.getenv("HF_TOKEN", None)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --------------------------------------------------------------------
# Training constants
# --------------------------------------------------------------------
FLAG_TEST = False              # If True, downsample data/epochs for debug
MAX_LENGTH = 96 if FLAG_TEST else 128
SEED = 1991
N_EPOCHS = 3 if FLAG_TEST else 50
EARLY_STOPPING_PATIENCE = 3
LEARNING_RATE = 2e-4
BATCH_SIZE = 1 if FLAG_TEST else 12
THRESH = 0.5  # default global threshold (overridden by per-label if available)

# --------------------------------------------------------------------
# Seeds & workers
# --------------------------------------------------------------------
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
random.seed(SEED)
set_seed(SEED)

def seed_worker(worker_id):
    np.random.seed(SEED + worker_id)
    random.seed(SEED + worker_id)

# --------------------------------------------------------------------
# Data utilities
# --------------------------------------------------------------------
def load_data(file_path: str) -> pd.DataFrame:
    """
    Read CSV with columns:
      - 'comment'
      - 'Autorrelato' (0/1)
      - 'Sintomas / Efeitos Colaterais' (0/1)
    Builds 'labels' as lists [Autorrelato, Sintomas].
    """
    logger.info(f"Loading data from file: {file_path}")
    df = pd.read_csv(file_path, usecols=['comment', 'Autorrelato', 'Sintomas / Efeitos Colaterais'])
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)

    if FLAG_TEST:
        df = df.sample(100, random_state=42)

    df['Autorrelato'] = df['Autorrelato'].astype(int).clip(0, 1)
    df['Sintomas / Efeitos Colaterais'] = df['Sintomas / Efeitos Colaterais'].astype(int).clip(0, 1)

    df['labels'] = df.apply(
        lambda r: [int(r['Autorrelato']), int(r['Sintomas / Efeitos Colaterais'])],
        axis=1
    )
    logger.info(f"Data loaded. Total samples: {len(df)}")
    return df


def cleanup_checkpoints(path: str):
    """Remove checkpoints directory to free disk space."""
    try:
        shutil.rmtree(path, ignore_errors=True)
        logger.info(f"Checkpoints removed at: {path}")
    except Exception as e:
        logger.warning(f"Failed to remove {path}: {e}")


def map_labels(df: pd.DataFrame):
    """
    Validate multilabel vector shape and return:
      - df (copy)
      - num_labels
      - label_names (human-friendly names in vector order)
    """
    df = df.copy()
    Y = np.vstack(df['labels'].values).astype(int)
    assert Y.shape[1] == 2, f"Expected 2 labels, got {Y.shape[1]}"
    label_names = ['Autorrelato', 'Sintomas / Efeitos Colaterais']
    pos_counts = Y.sum(axis=0)
    for i, lab in enumerate(label_names):
        logger.info(f"Label '{lab}' (idx {i}) - positives: {int(pos_counts[i])}")
    return df, Y.shape[1], label_names

# --------------------------------------------------------------------
# Threshold tuning utilities
# --------------------------------------------------------------------
def _sigmoid(x: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-x))

def tune_thresholds_by_f1(y_true: np.ndarray, y_prob: np.ndarray,
                          floor: float = 0.05, ceil: float = 0.95) -> np.ndarray:
    """
    Define a per-label threshold by maximizing F1 on validation.
    """
    L = y_true.shape[1]
    ths = np.full(L, 0.5, dtype=float)
    for j in range(L):
        yj = y_true[:, j]
        pj = y_prob[:, j]
        if yj.sum() == 0:
            ths[j] = 0.5
            continue
        p, r, t = precision_recall_curve(yj, pj)
        f1 = np.where((p + r) > 0, 2 * p * r / (p + r), 0.0)
        if len(t) == 0:
            ths[j] = 0.5
            continue
        f1_aligned = f1[1:]
        best_idx = int(np.argmax(f1_aligned))
        th_j = float(t[best_idx])
        ths[j] = float(np.clip(th_j, floor, ceil))
    return ths

def save_thresholds(path: str, thresholds: np.ndarray) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump({"thresholds": thresholds.tolist()}, f)


def load_thresholds(path: str) -> np.ndarray:
    with open(path, "r", encoding="utf-8") as f:
        return np.array(json.load(f)["thresholds"], dtype=float)

def _collect_logits_and_labels(model, eval_dataset, batch_size: int = BATCH_SIZE):
    """
    Collect raw logits and labels from an HF Dataset via DataLoader.
    Returns:
      - logits: [N, L]
      - labels: [N, L]
    """
    model.eval()
    dl = DataLoader(eval_dataset, batch_size=batch_size, shuffle=False, collate_fn=default_data_collator)
    logits_list, labels_list = [], []
    with torch.no_grad():
        for batch in dl:
            labels = batch["labels"]
            if isinstance(labels, torch.Tensor):
                labels_np = labels.numpy()
            else:
                labels_np = np.array(labels)
            labels_list.append(labels_np)

            batch = {k: v.to(device) for k, v in batch.items() if k != "labels"}
            out = model(**batch)
            logits_list.append(out.logits.detach().cpu().numpy())
    logits = np.concatenate(logits_list, axis=0)
    labels = np.concatenate(labels_list, axis=0)
    return logits, labels


In [ ]:
# Project paths and output folders (generic)
PROJECT_ROOT = os.path.abspath(os.getcwd())
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results')
SAVED_MODEL_DIR = os.path.join(PROJECT_ROOT, 'saved_model')
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(SAVED_MODEL_DIR, exist_ok=True)

# English aliases for Portuguese dataset label columns (kept original names for compatibility)
LABEL_NAMES_EN = ['Self-report', 'Symptoms / Side Effects']
LABEL_ALIAS = {'Autorrelato': 'Self-report', 'Sintomas / Efeitos Colaterais': 'Symptoms / Side Effects'}

# Note about label names

The dataset uses Portuguese column names for the two binary labels (`Autorrelato`, `Sintomas / Efeitos Colaterais`). For readability, `LABEL_NAMES_EN` and `LABEL_ALIAS` provide English equivalents while the code retains the original column names for compatibility across notebooks.

# Test

This script runs inference with a two-label multilabel classifier
— ["Autorrelato", "Sintomas / Efeitos Colaterais"] — using:

- 4-bit (NF4) quantized model + FP16 (avoids unsupported BFloat16 errors)
- PEFT/LoRA support (if an adapter_config.json is present) or a merged final model
- Sigmoid + per-label thresholds (loaded from ./saved_model/thresholds.json; fallback 0.5)

Two usage modes are implemented in the same file:

1) EXAMPLE MODE (small in-memory lists)
   - Function: `classify_and_save(...)`
   - Input: list of strings (e.g., `examples = [...]`)
   - Output: CSV with one row per text + 4 new prediction columns:
       probs_Autorrelato, pred_Autorrelato,
       probs_Sintomas_Efeitos_Colaterais, pred_Sintomas_Efeitos_Colaterais
   - Useful for quick smoke tests and debugging.

2) REAL-WORLD CSV MODE (large volume with chunks, tqdm and resume)
   - Function/entrypoint: `classify_file(...)`
   - Reads `prediction/final_comments_match_cleaned.csv` in chunks (default: 100k rows)
   - Required text column: `comment` (configurable via TEXT_COL)
   - Shows a progress bar (tqdm) and tracks previously processed rows
   - After each chunk:
       - Appends results to `prediction/final_comments_match_cleaned_pred.csv`
       - Updates a small state file `*.state.json` with the "next chunk"
         → In case of interruption, rerunning the script with the same paths
           resumes exactly where it left off without duplicating rows.
   - Keeps ALL original CSV columns and adds exactly 4 columns:
       probs_Autorrelato, pred_Autorrelato,
       probs_Sintomas_Efeitos_Colaterais, pred_Sintomas_Efeitos_Colaterais

Implementation details
----------------------
- Quantization: BitsAndBytes (NF4) 4-bit + `torch.float16` compute dtype
- Padding: ensures `pad_token` and `pad_token_id` are set (Llama-friendly for batch>1)
- Thresholds:
    - Loaded from `./saved_model/thresholds.json` (key "thresholds", per-label vector)
    - If missing or invalid, falls back to 0.5 per label
    - No retuning here (only application of saved thresholds)
- LoRA:
    - If `adapter_config.json` exists in `MODEL_DIR`, the base is loaded in 4-bit
      and the saved PEFT adapter is attached.
    - Tip: for production, consider merging LoRA into the base model (merge_and_unload)
      and point `MODEL_DIR = "./saved_model_merged"` to avoid classifier-head warnings.

Quick run (example)
-------------------
- In-memory test:
    - Edit the `examples` list in the `if __name__ == "__main__":` block
    - This generates `example_predictions.csv`

- Process the real dataset:
    - Ensure `prediction/final_comments_match_cleaned.csv` exists
    - Adjust constants if needed (BATCH_SIZE, CHUNK_ROWS, TEXT_COL)
    - Run the script (it will create/update `prediction/final_comments_match_cleaned_pred.csv`)
    - If interrupted, rerun to continue from the last completed chunk.

Added columns
-------------
- `probs_Autorrelato` (float)
- `pred_Autorrelato`  (0/1)
- `probs_Sintomas_Efeitos_Colaterais` (float)
- `pred_Sintomas_Efeitos_Colaterais`  (0/1)

Best practices
--------------
- Reduce `BATCH_SIZE` and/or `MAX_LENGTH` if VRAM is insufficient.
- For very large datasets prefer Parquet for speed and storage efficiency.
- Keep `MODEL_DIR` stable across runs for consistent resumption.
- If LoRA is merged, inference simplifies and warnings disappear.


In [ ]:
# -*- coding: utf-8 -*-
"""
Multilabel classifier inference (thresholded, no temperature scaling).

- Loads saved model dir (supports PEFT/LoRA adapters) in 4-bit NF4.
- Forces FP16 to avoid 'unsupported BFloat16' errors.
- Ensures padding token is set (fixes batch>1 errors for Llama).
- Predicts probabilities via sigmoid and per-label thresholds (from thresholds.json or 0.5 fallback).
- Works with a list of texts OR a large pandas DataFrame (chunked, memory-friendly).
- Prints human-readable class names and saves results to CSV.

Label names are explicitly set to:
  ["Autorrelato", "Sintomas / Efeitos Colaterais"]
so output columns and prints use these names.

TIP: If you have a merged model (LoRA merged into base, including classifier head),
     set MODEL_DIR to that folder (e.g., './saved_model_merged') to avoid 'score.weight' warnings.
"""

import os
import json
from typing import List, Optional, Sequence, Union

import numpy as np
import pandas as pd
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
)
from peft import PeftModel  # pip install peft

# -----------------------------
# Global settings
# -----------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_LENGTH = 128
BATCH_SIZE = 32
MODEL_DIR = "./saved_model"  # If you have a merged model, point this to './saved_model_merged'

# Desired, human-readable label names in the correct id order.
DESIRED_LABEL_NAMES = ["Autorrelato", "Sintomas / Efeitos Colaterais"]


# -----------------------------
# Helpers
# -----------------------------
def _sigmoid(x: np.ndarray) -> np.ndarray:
    """Vectorized sigmoid for numpy arrays."""
    return 1.0 / (1.0 + np.exp(-x))


def _ensure_padding(tokenizer, model) -> None:
    """
    Ensure we have a pad token and pad_token_id so batches > 1 work with Llama.
    Uses EOS as PAD if no explicit pad token exists.
    """
    if tokenizer.pad_token is None:
        if tokenizer.eos_token is None:
            tokenizer.add_special_tokens({"pad_token": "[PAD]"})
            model.resize_token_embeddings(len(tokenizer))
        else:
            tokenizer.pad_token = tokenizer.eos_token

    if getattr(model.config, "pad_token_id", None) is None:
        model.config.pad_token_id = tokenizer.pad_token_id

    tokenizer.padding_side = "right"


def load_thresholds_or_default(path: str, num_labels: int, default: float = 0.5) -> np.ndarray:
    """
    Load per-label thresholds from a JSON file. If missing or malformed,
    returns a vector filled with `default`.
    """
    try:
        with open(path, "r", encoding="utf-8") as f:
            ths = np.array(json.load(f)["thresholds"], dtype=float)
        if ths.shape[0] != num_labels:
            ths = np.full(num_labels, default, dtype=float)
    except Exception:
        ths = np.full(num_labels, default, dtype=float)
    return ths


def load_model_and_tokenizer(model_dir: str = MODEL_DIR):
    """
    Load tokenizer and model in 4-bit (NF4) and attach PEFT/LoRA adapter if present.
    Forces FP16 (safe across GPUs) to avoid 'unsupported BFloat16'.
    Also ensures padding and sets human-readable label names.
    """
    tokenizer = AutoTokenizer.from_pretrained(model_dir, use_fast=True)

    # Force FP16 compute dtype (robust for environments without bf16 support)
    compute_dtype = torch.float16
    bnb_4bit = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype,
    )

    adapter_cfg_path = os.path.join(model_dir, "adapter_config.json")
    if os.path.exists(adapter_cfg_path):
        # PEFT adapter path: load the base in 4-bit and attach adapter
        with open(adapter_cfg_path, "r", encoding="utf-8") as f:
            adapter_cfg = json.load(f)
        base = adapter_cfg.get("base_model_name_or_path")
        if not base:
            raise ValueError("adapter_config.json missing 'base_model_name_or_path'.")

        base_model = AutoModelForSequenceClassification.from_pretrained(
            base,
            quantization_config=bnb_4bit,
            device_map="auto",         # auto placement (offload if needed)
            torch_dtype=compute_dtype, # enforce fp16
        )
        model = PeftModel.from_pretrained(base_model, model_dir)
    else:
        # Full fine-tuned (possibly merged) model saved in this folder: load in 4-bit
        model = AutoModelForSequenceClassification.from_pretrained(
            model_dir,
            quantization_config=bnb_4bit,
            device_map="auto",
            torch_dtype=compute_dtype,
        )

    _ensure_padding(tokenizer, model)
    model.eval()

    # Force desired label names (assumes id 0 = Autorrelato, id 1 = Sintomas/Efeitos)
    num_labels = model.config.num_labels
    if len(DESIRED_LABEL_NAMES) != num_labels:
        raise ValueError(
            f"DESIRED_LABEL_NAMES length ({len(DESIRED_LABEL_NAMES)}) "
            f"does not match num_labels ({num_labels})."
        )
    label_names = DESIRED_LABEL_NAMES
    model.config.id2label = {i: name for i, name in enumerate(label_names)}
    model.config.label2id = {name: i for i, name in enumerate(label_names)}

    return model, tokenizer, label_names


# -----------------------------
# Core prediction routines
# -----------------------------
@torch.no_grad()
def predict_multilabel_texts(
    texts: Sequence[str],
    model,
    tokenizer,
    label_names: List[str],
    thresholds: np.ndarray,
    batch_size: int = BATCH_SIZE,
    max_length: int = MAX_LENGTH,
) -> pd.DataFrame:
    """
    Predict probabilities and thresholded labels for a list of strings.
    Returns a DataFrame with one row per text, including per-label columns
    based on `label_names`.
    """
    if isinstance(texts, str):
        texts = [texts]

    # Safety: if padding is still missing for some reason, force batch_size=1
    if getattr(model.config, "pad_token_id", None) is None and batch_size > 1:
        batch_size = 1

    probs_list = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        inputs = tokenizer(
            batch, return_tensors="pt", padding=True, truncation=True, max_length=max_length
        )
        inputs = {k: v.to(DEVICE, non_blocking=True) for k, v in inputs.items()}
        logits_t = model(**inputs).logits.detach()       # [B, L]
        probs_t  = torch.sigmoid(logits_t)               # [B, L]
        probs    = probs_t.cpu().numpy().astype("float32")
        probs_list.append(probs)

    probs = np.vstack(probs_list)      # [N, L]
    thr = thresholds.reshape(1, -1)    # [1, L]
    preds = (probs >= thr).astype(int) # [N, L]

    rows = []
    for text, p_row, y_row in zip(texts, probs, preds):
        row = {"text": text}
        for j, name in enumerate(label_names):
            pcol = f"probs_{name}".replace(" ", "_").replace("/", "_")
            ycol = f"pred_{name}".replace(" ", "_").replace("/", "_")
            row[pcol] = float(p_row[j])
            row[ycol] = int(y_row[j])
        rows.append(row)
    return pd.DataFrame(rows)


@torch.no_grad()
def predict_multilabel_dataframe(
    df: pd.DataFrame,
    text_col: str,
    model,
    tokenizer,
    label_names: List[str],
    thresholds: np.ndarray,
    out_csv: Optional[str] = None,
    batch_size: int = BATCH_SIZE,
    max_length: int = MAX_LENGTH,
    write_mode: str = "w",
    header: bool = True,
    chunk_rows: int = 100_000,
) -> Optional[pd.DataFrame]:
    """
    Scalable inference for very large DataFrames.
    Processes in row chunks to bound memory, predicts in mini-batches,
    and optionally appends results to CSV.
    """
    assert text_col in df.columns, f"Column '{text_col}' not found."

    collector = [] if out_csv is None else None
    n = len(df)

    for start in range(0, n, chunk_rows):
        end = min(start + chunk_rows, n)
        sub = df.iloc[start:end]
        texts = sub[text_col].astype(str).tolist()

        chunk_rows_out = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            inputs = tokenizer(
                batch, return_tensors="pt", padding=True, truncation=True, max_length=max_length
            )
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
            logits_t = model(**inputs).logits.detach()      # tensor
            probs_t  = torch.sigmoid(logits_t)              
            probs    = probs_t.cpu().numpy()               
            thr = thresholds.reshape(1, -1)
            preds = (probs >= thr).astype(int)

            for text, p_row, y_row in zip(batch, probs, preds):
                row = {"text": text}
                for j, name in enumerate(label_names):
                    pcol = f"probs_{name}".replace(" ", "_").replace("/", "_")
                    ycol = f"pred_{name}".replace(" ", "_").replace("/", "_")
                    row[pcol] = float(p_row[j])
                    row[ycol] = int(y_row[j])
                chunk_rows_out.append(row)

        chunk_df = pd.DataFrame(chunk_rows_out)

        if out_csv is not None:
            chunk_df.to_csv(out_csv, mode=write_mode, header=header, index=False, encoding="utf-8")
            write_mode, header = "a", False
        else:
            collector.append(chunk_df)

    if collector is not None:
        return pd.concat(collector, ignore_index=True)
    return None


# -----------------------------
# High-level convenience APIs
# -----------------------------
def classify_and_save(
    texts: Union[str, Sequence[str]],
    out_csv: str = "example_predictions.csv",
    model_dir: str = MODEL_DIR,
) -> pd.DataFrame:
    """
    Convenience wrapper for small inputs (list/str). Saves to CSV and returns the DataFrame.
    """
    model, tokenizer, label_names = load_model_and_tokenizer(model_dir)
    thresholds = load_thresholds_or_default(
        os.path.join(model_dir, "thresholds.json"),
        num_labels=len(label_names),
        default=0.5,
    )

    # Print class names for visibility
    print("Class names:", label_names)

    df = predict_multilabel_texts(texts, model, tokenizer, label_names, thresholds)
    df.to_csv(out_csv, index=False, encoding="utf-8")
    return df


def classify_dataframe_and_save(
    df: pd.DataFrame,
    text_col: str,
    out_csv: str,
    model_dir: str = MODEL_DIR,
    chunk_rows: int = 200_000,
    batch_size: int = BATCH_SIZE,
) -> None:
    """
    Scalable wrapper for very large datasets in a DataFrame.
    Streams through the DataFrame in chunks and writes to CSV incrementally.
    Suitable for millions to tens of millions of comments (I/O bound).
    """
    model, tokenizer, label_names = load_model_and_tokenizer(model_dir)
    thresholds = load_thresholds_or_default(
        os.path.join(model_dir, "thresholds.json"),
        num_labels=len(label_names),
        default=0.5,
    )

    print("Class names:", label_names)

    predict_multilabel_dataframe(
        df=df,
        text_col=text_col,
        model=model,
        tokenizer=tokenizer,
        label_names=label_names,
        thresholds=thresholds,
        out_csv=out_csv,
        batch_size=batch_size,
        chunk_rows=chunk_rows,
        write_mode="w",
        header=True,
    )


# -----------------------------
# Examples
# -----------------------------
if __name__ == "__main__":
    examples = [
        "Comecei a usar dura faz 3 semanas e percebi que minha libido caiu.",
        "Meu amigo disse que só teve mais suor à noite depois de usar.",
        "Tô pensando em começar, mas tenho medo de cair cabelo.",
        "Já usei oxan e não senti nada demais.",
        "Depois da aplicação fiquei com dor forte no local.",
        "Não usei nada ainda, mas fico inseguro com esses relatos.",
        "Com 22 anos usei stano e enchi de espinhas.",
        "A disposição aumentou muito depois que iniciei.",
        "Usei 4 semanas e meu cabelo começou a cair.",
        "Nunca tomei, mas penso em resolver minha falta de energia assim.",
        "Fiz TRT por 6 meses, controlei hematócrito e deu tudo certo.",
        "Meu parceiro ficou muito agressivo quando começou o ciclo.",
        "Depois da primeira aplicação já senti pump absurdo.",
        "Usei trembo e tive insônia todas as noites.",
        "Minha prima disse que um colega dela desenvolveu gineco.",
        "Já fiz vários ciclos e nunca tive problema.",
        "No começo tive azia e dor de estômago, depois passou.",
        "Um amigo relatou que ficou ansioso demais usando.",
        "Depois do ciclo perdi tudo e fiquei desanimado.",
        "Ainda não usei, mas quero fazer TPC direitinho quando usar.",
        "Testei oxan e parecia farinha, não senti nada.",
        "Tenho acne mesmo sem usar nada, imagina quem usa.",
        "Com enantato meu sono piorou bastante.",
        "Minha irmã contou que o namorado dela ficou impotente depois.",
        "Fiz 2 ciclos e só dor de cabeça de vez em quando.",
        "Vi muitos casos de depressão depois do uso.",
        "No dia da aplicação tive febre e calafrio.",
        "Tenho medo de prejudicar meu fígado com isso.",
        "Fiz um ciclo leve e não tive nenhum colateral.",
        "Desisti quando vi relatos de muita retenção de líquido."
    ]

    df = classify_and_save(examples, out_csv="example_predictions.csv", model_dir=MODEL_DIR)

    print("\n=== Results (with per-label thresholds) ===")
    model, _, label_names = load_model_and_tokenizer(MODEL_DIR)
    thresholds = load_thresholds_or_default(
        os.path.join(MODEL_DIR, "thresholds.json"),
        num_labels=len(label_names),
        default=0.5,
    )
    print("Class names:", label_names)
    print("Per-label thresholds:", {name: float(t) for name, t in zip(label_names, thresholds)})

    for _, row in df.iterrows():
        print("\nText:", row["text"])
        for name in label_names:  # use the same label_names to match columns
            pcol = f"probs_{name}".replace(" ", "_").replace("/", "_")
            ycol = f"pred_{name}".replace(" ", "_").replace("/", "_")
            print(f"  {name:>30s} | prob = {row[pcol]:.4f} | class = {row[ycol]}")
    print("\nSaved file: example_predictions.csv")


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at meta-llama/Meta-Llama-3.1-8B and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Class names: ['Autorrelato', 'Sintomas / Efeitos Colaterais']

=== Results (with per-label thresholds) ===


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at meta-llama/Meta-Llama-3.1-8B and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Class names: ['Autorrelato', 'Sintomas / Efeitos Colaterais']
Per-label thresholds: {'Autorrelato': 0.2107662856578827, 'Sintomas / Efeitos Colaterais': 0.66888028383255}

Text: Comecei a usar dura faz 3 semanas e percebi que minha libido caiu.
                     Autorrelato | prob = 0.9971 | class = 1
   Sintomas / Efeitos Colaterais | prob = 0.9849 | class = 1

Text: Meu amigo disse que só teve mais suor à noite depois de usar.
                     Autorrelato | prob = 0.0060 | class = 0
   Sintomas / Efeitos Colaterais | prob = 0.7852 | class = 1

Text: Tô pensando em começar, mas tenho medo de cair cabelo.
                     Autorrelato | prob = 0.0000 | class = 0
   Sintomas / Efeitos Colaterais | prob = 0.6558 | class = 0

Text: Já usei oxan e não senti nada demais.
                     Autorrelato | prob = 0.5190 | class = 1
   Sintomas / Efeitos Colaterais | prob = 0.0303 | class = 0

Text: Depois da aplicação fiquei com dor forte no local.
                     Autorrelato 

# Full data

In [ ]:
# -*- coding: utf-8 -*-
"""
Batch multilabel inference on a large CSV with progress (tqdm) and resume support.
- Reads input CSV in chunks from `prediction/final_comments_match_cleaned.csv` (text column: 'comment')
- Keeps all original columns and appends 4 prediction columns:
    probs_Autorrelato, pred_Autorrelato,
    probs_Sintomas_Efeitos_Colaterais, pred_Sintomas_Efeitos_Colaterais
- Applies per-label thresholds loaded from ./saved_model/thresholds.json (fallback 0.5)
- 4-bit NF4 + FP16 + optional PEFT adapter
- Resume-safe: writes a small state file to continue after interruptions
"""

import os, json, tempfile
os.environ["TOKENIZERS_PARALLELISM"] = "true"
from typing import List
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

import torch
torch.backends.cuda.matmul.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

from transformers import AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig
from peft import PeftModel

# -----------------------------
# Paths & settings
# -----------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_LENGTH = 128
BATCH_SIZE = 512
CHUNK_ROWS = 5000



MODEL_DIR  = "./saved_model"  # or "./saved_model_merged" if you merged LoRA
INPUT_FILE = "prediction/final_comments_match_cleaned.csv"
OUTPUT_FILE = "prediction/final_comments_match_cleaned_pred.csv"
STATE_FILE  = OUTPUT_FILE + ".state.json"
TEXT_COL = "comment"

LABEL_NAMES = ["Autorrelato", "Sintomas / Efeitos Colaterais"]

# -----------------------------
# Helpers
# -----------------------------
def _sigmoid(x: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-x))

def _ensure_padding(tokenizer, model):
    if tokenizer.pad_token is None:
        if tokenizer.eos_token is None:
            tokenizer.add_special_tokens({"pad_token": "[PAD]"})
            model.resize_token_embeddings(len(tokenizer))
        else:
            tokenizer.pad_token = tokenizer.eos_token
    if getattr(model.config, "pad_token_id", None) is None:
        model.config.pad_token_id = tokenizer.pad_token_id
    tokenizer.padding_side = "right"

def load_thresholds(path, num_labels, default=0.5):
    try:
        with open(path, "r", encoding="utf-8") as f:
            ths = np.array(json.load(f)["thresholds"], dtype=float)
        if ths.shape[0] != num_labels:
            ths = np.full(num_labels, default, dtype=float)
    except Exception:
        ths = np.full(num_labels, default, dtype=float)
    return ths


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at meta-llama/Meta-Llama-3.1-8B and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[RESUME] Continuing from chunk 23


Processing:   0%|          | 0/123 [00:00<?, ?chunk/s]